In [2]:
import gradio as gr
import numpy as np
import joblib
import json
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image


# ================================
# Load Models & Class Labels
# ================================
cnn_model = load_model("/content/drive/MyDrive/emotion_models/cnn_emotion_model.h5")
rf_model = joblib.load("/content/drive/MyDrive/emotion_models/rf_emotion_model.pkl")
svm_model = joblib.load("/content/drive/MyDrive/emotion_models/svm_emotion_model.pkl")

with open("/content/drive/MyDrive/emotion_models/class_indices.json", "r") as f:
    class_labels = json.load(f)  # {"0":"angry", "1":"happy", "2":"sad"}


# ================================
# Preprocessing Function
# ================================
def preprocess_image(img_path, target_size=(128, 128)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img) / 255.0
    return img_array


# ================================
# Prediction Functions
# ================================
def predict_emotion(img, model_choice):
    # Convert to array
    img_array = preprocess_image(img, target_size=(128,128))
    img_array_expanded = np.expand_dims(img_array, axis=0)

    if model_choice == "CNN":
        prediction = cnn_model.predict(img_array_expanded)
        class_index = np.argmax(prediction[0])

    elif model_choice == "Random Forest":
        flat_img = img_array.flatten().reshape(1, -1)
        prediction = rf_model.predict(flat_img)
        class_index = prediction[0]

    elif model_choice == "SVM":
        flat_img = img_array.flatten().reshape(1, -1)
        prediction = svm_model.predict(flat_img)
        class_index = prediction[0]

    else:
        return "Invalid model selected"

    return f"Predicted Emotion: {class_labels[str(class_index)]}"


# ================================
# Gradio Interface
# ================================
demo = gr.Interface(
    fn=predict_emotion,
    inputs=[
        gr.Image(type="filepath", label="Upload a Face Image"),
        gr.Radio(["CNN", "Random Forest", "SVM"], label="Choose Model")
    ],
    outputs="text",
    title="Facial Emotion Detection",
    description="Upload a face image and select a model (CNN, Random Forest, or SVM) to predict the emotion."
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://836cfb150f82c12ef7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
